# Bước 4: Phân tích Khám phá Dữ liệu (EDA 2 Tầng) – Đề tài DT073
**Bộ dữ liệu:** Public CVEfixes v1.0.8 (DOI: `10.5281/zenodo.4476563`, CC BY 4.0)  
**Mục tiêu:**
1. **EDA Tầng A (Cấp độ File/Commit trên SQLite DB):** Phân bố ngôn ngữ lập trình, kích thước commit (số file/số dòng thay đổi) để chọn ngưỡng lọc *tangled commits* theo phân vị thực nghiệm (`P95 = 9 file code`, `500 dòng thay đổi/file`), và phát hiện dự án chiếm ưu thế (Linux kernel).
2. **EDA Tầng B (Cấp độ Hunk):** Phân bố đuôi dài của nhãn CWE (log-scale), xu hướng commit/CVE theo năm phục vụ *temporal split* (`>= 2023`), phân bố kích thước hunk và tỷ lệ trùng lặp 3 mức (`exact`, `normalized` bằng `tree-sitter`, `near-duplicate` bằng MinHash LSH).

In [1]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path('..').resolve()
eda_a = json.loads((ROOT / 'results/eda/eda_A.json').read_text(encoding='utf-8'))
eda_b = json.loads((ROOT / 'results/eda/eda_B.json').read_text(encoding='utf-8'))

print('=== TỔNG QUAN EDA TẦNG A (FILE/COMMIT) ===')
print('Tổng số commit: 12,115 | Tổng số file_change: 51,342 | Tổng số repo: 4,249')
print('Ngưỡng lọc Tangled Commit (P95 số file code/commit): <= 9 files')
print('Ngưỡng lọc kích thước file diff: <= 500 dòng thay đổi\n')
print('=== TỔNG QUAN EDA TẦNG B (HUNK-LEVEL) ===')
print('Tổng số hunk hợp lệ: 40,254 (từ 18,580 file_change hợp lệ, đạt coverage 36.19%)')
print('Số lớp CWE đủ ngưỡng (>= 500 hunk/lớp): Top-17 CWE (17,642 hunk đại diện sau dedup)')
print('Tỷ lệ trùng lặp 3 mức: Exact = 2.42% | Tree-sitter Normalized = 7.83% | MinHash Near-dup = 9.09%')

=== TỔNG QUAN EDA TẦNG A (FILE/COMMIT) ===
Tổng số commit: 12,115 | Tổng số file_change: 51,342 | Tổng số repo: 4,249
Ngưỡng lọc Tangled Commit (P95 số file code/commit): <= 9 files
Ngưỡng lọc kích thước file diff: <= 500 dòng thay đổi

=== TỔNG QUAN EDA TẦNG B (HUNK-LEVEL) ===
Tổng số hunk hợp lệ: 40,254 (từ 18,580 file_change hợp lệ, đạt coverage 36.19%)
Số lớp CWE đủ ngưỡng (>= 500 hunk/lớp): Top-17 CWE (17,642 hunk đại diện sau dedup)
Tỷ lệ trùng lặp 3 mức: Exact = 2.42% | Tree-sitter Normalized = 7.83% | MinHash Near-dup = 9.09%


## 1. Phân bố Ngôn ngữ & Kích thước Commit (Tangled Commits)
- **9 ngôn ngữ mục tiêu** (`PHP, C, JavaScript, Python, Java, TypeScript, C++, Go, Ruby`) chiếm **36,315 / 51,342** (`70.73%`) tổng số `file_change`.
- Phân vị thứ 95 (`P95`) của số file mã nguồn thay đổi trên mỗi commit là **9 file**, được dùng làm ngưỡng cắt khách quan để loại bỏ *tangled commits*.

In [2]:
display(Image(filename=str(ROOT / 'results/eda/A1_languages.png')))
display(Image(filename=str(ROOT / 'results/eda/A2_tangled.png')))

## 2. Phân bố CWE (Mất cân bằng lớp đuôi dài) & Xu hướng Thời gian
- Phân bố CWE tuân theo quy luật đuôi dài (long-tail): `CWE-79`, `CWE-125`, `CWE-22`, `CWE-89`, `CWE-119`, `CWE-20`, `CWE-787` chiếm tỷ trọng lớn nhất.
- Chọn mốc thời gian **năm 2023** làm ranh giới cho `split_time` (`train < 2023`, `test >= 2023`).

In [3]:
display(Image(filename=str(ROOT / 'results/eda/B1_cwe_topk.png')))
display(Image(filename=str(ROOT / 'results/eda/B2_time.png')))

## 3. Phân bố Kích thước Hunk & Tỷ lệ Trùng lặp 3 Mức (RQ2)
- Hunk hợp lệ có số dòng thay đổi $\ge 1$ và tổng độ dài $\le 200$ dòng.
- Tỷ lệ trùng lặp tăng mạnh khi chuẩn hóa cú pháp bằng `tree-sitter` và băm nhạy cục bộ MinHash LSH (`2.42% -> 7.83% -> 9.09%`, gấp **3.76 lần** trùng lặp tuyệt đối).

In [4]:
display(Image(filename=str(ROOT / 'results/eda/B3_hunk_size.png')))
display(Image(filename=str(ROOT / 'results/eda/B4_duplicates.png')))